In [1]:
# ==================== 单元格 1：环境准备（Hard 版本） ====================
!pip install -q pandas numpy matplotlib seaborn scikit-learn transformers datasets accelerate torch

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import time
import os
from tqdm.auto import tqdm


# 设置中文字体
!wget -O /usr/share/fonts/truetype/liberation/simhei.ttf "https://www.wfonts.com/download/data/2014/06/01/simhei/chinese.simhei.ttf" 2>/dev/null
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")

使用设备: cuda


In [2]:
# ==================== 单元格 2：Hard 难度数据加载与预处理 ====================
# ==================== Kaggle 单元格：数据加载 ====================
# -------------------- 1. 设置 Kaggle 数据路径 --------------------
data_dir = '/kaggle/input/datasets/victoriagyf/6980-scam-dataset/'

# 验证文件是否存在
for file in ['Hard_train.txt', 'Hard_dev.txt', 'Hard_test.txt']:
    full_path = os.path.join(data_dir, file)
    if not os.path.exists(full_path):
        print(f"警告：文件 {full_path} 不存在，请检查路径！")
    else:
        print(f"✅ 找到文件：{file}")

# -------------------- 2. 加载 Hard 数据 --------------------
def load_scamgen_txt(filepath):
    return pd.read_csv(filepath, sep='\t', header=None, names=['text', 'label'])

hard_train_df = load_scamgen_txt(data_dir + 'Hard_train.txt')
hard_val_df   = load_scamgen_txt(data_dir + 'Hard_dev.txt')
hard_test_df  = load_scamgen_txt(data_dir + 'Hard_test.txt')

print("Hard 训练集原始标签分布：\n", hard_train_df['label'].value_counts())

# -------------------- 2. 标签映射：0~3 → 1 (欺诈)，4 → 0 (正常) --------------------
def map_to_binary(label):
    return 0 if label == 4 else 1

for df in [hard_train_df, hard_val_df, hard_test_df]:
    df['label_id'] = df['label'].apply(map_to_binary)

print("\nHard 二分类训练集分布（0=正常，1=欺诈）：\n", hard_train_df['label_id'].value_counts())
print("Hard 二分类验证集分布：\n", hard_val_df['label_id'].value_counts())
print("Hard 二分类测试集分布：\n", hard_test_df['label_id'].value_counts())

# -------------------- 3. 特征提取（URL, Email, Phone）--------------------
email_pattern = re.compile(
    r"([a-z0-9!#$%&'*+/=?^_`{|}~-]+(?:\.[a-z0-9!#$%&'*+/=?^_`{|}~-]+)*(@|\sat\s)"
    r"(?:[a-z0-9](?:[a-z0-9-]*[a-z0-9])?(\.|\sdot\s))+[a-z0-9](?:[a-z0-9-]*[a-z0-9])?)",
    re.IGNORECASE
)
url_pattern = re.compile(
    r"http[s]?://(?:[a-zA-Z0-9]|[$-_@.&+]|[!*(),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+"
)
phone_pattern = re.compile(r"\b\d{10,}\b")

def url_check(text): return 1 if url_pattern.search(str(text)) else 0
def email_check(text): return 1 if email_pattern.search(str(text)) else 0
def phone_check(text): return 1 if phone_pattern.search(str(text)) else 0

for df in [hard_train_df, hard_val_df, hard_test_df]:
    df['has_url']   = df['text'].apply(url_check)
    df['has_email'] = df['text'].apply(email_check)
    df['has_phone'] = df['text'].apply(phone_check)

print("特征提取完成。")

# -------------------- 4. 准备模型输入数据（list & numpy array）--------------------
hard_train_texts = hard_train_df['text'].tolist()
hard_train_labels = hard_train_df['label_id'].tolist()
hard_train_feats = hard_train_df[['has_url', 'has_email', 'has_phone']].values.astype(np.float32)

hard_val_texts = hard_val_df['text'].tolist()
hard_val_labels = hard_val_df['label_id'].tolist()
hard_val_feats = hard_val_df[['has_url', 'has_email', 'has_phone']].values.astype(np.float32)

hard_test_texts = hard_test_df['text'].tolist()
hard_test_labels = hard_test_df['label_id'].tolist()
hard_test_feats = hard_test_df[['has_url', 'has_email', 'has_phone']].values.astype(np.float32)

print(f"\nHard 训练集: {len(hard_train_texts)} | 验证集: {len(hard_val_texts)} | 测试集: {len(hard_test_texts)}")

# -------------------- 5. 计算类别权重（解决不平衡）--------------------
hard_class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=hard_train_labels
)
hard_class_weights = torch.tensor(hard_class_weights, dtype=torch.float)
print("Hard 类别权重:", hard_class_weights)

# -------------------- 6. 示例查看 --------------------
print("\nHard 示例文本（前150字符）：")
print(hard_train_texts[0][:150])
print("标签：", hard_train_labels[0])

✅ 找到文件：Hard_train.txt
✅ 找到文件：Hard_dev.txt
✅ 找到文件：Hard_test.txt
Hard 训练集原始标签分布：
 label
0    20059
1    17278
3    15787
2    15250
4    13894
Name: count, dtype: int64

Hard 二分类训练集分布（0=正常，1=欺诈）：
 label_id
1    68374
0    13894
Name: count, dtype: int64
Hard 二分类验证集分布：
 label_id
1    5743
0    1294
Name: count, dtype: int64
Hard 二分类测试集分布：
 label_id
1    6153
0    1347
Name: count, dtype: int64
特征提取完成。

Hard 训练集: 82268 | 验证集: 7037 | 测试集: 7500
Hard 类别权重: tensor([2.9606, 0.6016])

Hard 示例文本（前150字符）：
小刘在吗？我是您同窗小陈 我找您有点事，太需要你的帮忙了 你能尽早给我转点吗 
标签： 1


In [3]:
# ==================== 单元格 3：Hard 版本的 Dataset & DataLoader ====================
MODEL_NAME = 'bert-base-uncased'   # 可改为 'bert-base-chinese' 提升中文效果
MAX_LEN = 128                      # 基于 Easy 的长度统计，Hard 应类似
BATCH_SIZE = 32                    # 根据 T4 16GB 显存设置
EPOCHS = 1                         # Hard 难度下 2 个 epoch 足够
LEARNING_RATE = 2e-5
FEATURE_DIM = 3                    # has_url, has_email, has_phone
NUM_LABELS = 2                     # 二分类

tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

class SMSDatasetWithFeatures(Dataset):
    def __init__(self, texts, labels, features, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.features = features
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        feats = self.features[idx]

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'features': torch.tensor(feats, dtype=torch.float),
            'labels': torch.tensor(label, dtype=torch.long)
        }

hard_train_dataset = SMSDatasetWithFeatures(hard_train_texts, hard_train_labels, hard_train_feats, tokenizer, MAX_LEN)
hard_val_dataset   = SMSDatasetWithFeatures(hard_val_texts, hard_val_labels, hard_val_feats, tokenizer, MAX_LEN)
hard_test_dataset  = SMSDatasetWithFeatures(hard_test_texts, hard_test_labels, hard_test_feats, tokenizer, MAX_LEN)

# 为避免多进程报错，num_workers 设为 0（Colab 中较稳定）
hard_train_loader = DataLoader(hard_train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
hard_val_loader   = DataLoader(hard_val_dataset, batch_size=BATCH_SIZE, num_workers=0)
hard_test_loader  = DataLoader(hard_test_dataset, batch_size=BATCH_SIZE, num_workers=0)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [4]:
# ==================== 单元格 4：模型定义 ====================
class BertWithFeatures(nn.Module):
    def __init__(self, model_name, feature_dim, num_labels):
        super(BertWithFeatures, self).__init__()
        self.bert = BertModel.from_pretrained(model_name)
        self.feature_fc = nn.Linear(feature_dim, 32)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.bert.config.hidden_size + 32, num_labels)

    def forward(self, input_ids, attention_mask, features):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0, :]
        feature_embedding = self.feature_fc(features)
        combined = torch.cat((cls_embedding, feature_embedding), dim=1)
        combined = self.dropout(combined)
        logits = self.classifier(combined)
        return logits

hard_model = BertWithFeatures(MODEL_NAME, FEATURE_DIM, NUM_LABELS)
hard_model.to(device)

optimizer = AdamW(hard_model.parameters(), lr=LEARNING_RATE)
total_steps = len(hard_train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)
hard_loss_fn = nn.CrossEntropyLoss(weight=hard_class_weights.to(device))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
# ==================== 单元格 5：训练 & 评估函数 ====================
def train_epoch(model, data_loader, optimizer, scheduler, loss_fn, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    for batch in tqdm(data_loader, desc='Training'):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        features = batch['features'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask, features)
        loss = loss_fn(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item() * labels.size(0)
        _, preds = torch.max(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total

def eval_model(model, data_loader, loss_fn, device):
    model.eval()
    total_loss = 0.0
    total = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for batch in tqdm(data_loader, desc='Evaluating'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            features = batch['features'].to(device)
            labels = batch['labels'].to(device)

            logits = model(input_ids, attention_mask, features)
            loss = loss_fn(logits, labels)
            total_loss += loss.item() * labels.size(0)
            total += labels.size(0)

            _, preds = torch.max(logits, dim=1)
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    avg_loss = total_loss / total
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='binary')
    prec = precision_score(all_labels, all_preds, average='binary')
    rec = recall_score(all_labels, all_preds, average='binary')

    return {
        'loss': avg_loss,
        'accuracy': acc,
        'f1': f1,
        'precision': prec,
        'recall': rec,
        'preds': all_preds,
        'labels': all_labels
    }

In [6]:
!ls -R /kaggle/working

/kaggle/working:
__notebook__.ipynb


In [7]:
# ==================== 单元格 6：Hard 模型训练 ====================
print("开始训练 Hard 模型...")
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    train_loss, train_acc = train_epoch(hard_model, hard_train_loader, optimizer, scheduler, hard_loss_fn, device)
    print(f"训练 Loss: {train_loss:.4f} | 训练 Acc: {train_acc:.4f}")

    val_metrics = eval_model(hard_model, hard_val_loader, hard_loss_fn, device)
    print(f"验证 Loss: {val_metrics['loss']:.4f} | Acc: {val_metrics['accuracy']:.4f} | F1: {val_metrics['f1']:.4f}")

print("\nHard 模型训练完成！")

开始训练 Hard 模型...

Epoch 1/1


Training:   0%|          | 0/2571 [00:00<?, ?it/s]

训练 Loss: 0.0538 | 训练 Acc: 0.9842


Evaluating:   0%|          | 0/220 [00:00<?, ?it/s]

验证 Loss: 2.6342 | Acc: 0.8752 | F1: 0.9266

Hard 模型训练完成！


In [8]:
# ==================== 单元格 7：Hard 测试集评估 & 保存模型（Kaggle 版）====================
# 评估测试集
hard_test_metrics = eval_model(hard_model, hard_test_loader, hard_loss_fn, device)
print("Hard 测试集结果：")
print(f"准确率: {hard_test_metrics['accuracy']:.4f}")
print(f"F1 值:  {hard_test_metrics['f1']:.4f}")
print(f"召回率: {hard_test_metrics['recall']:.4f}")
print(f"精确率: {hard_test_metrics['precision']:.4f}")

# 保存模型到 Kaggle 工作目录（/kaggle/working）
save_path = '/kaggle/working/scam_fraud_model_hard'
os.makedirs(save_path, exist_ok=True)

# 保存模型权重
model_save_path = os.path.join(save_path, 'pytorch_model.bin')
torch.save(hard_model.state_dict(), model_save_path)
print(f"模型权重已保存至 {model_save_path}")

# 保存分词器
tokenizer.save_pretrained(save_path)
print(f"分词器已保存至 {save_path}")

# 验证保存的文件
print("\n保存的文件列表：")
for file in os.listdir(save_path):
    print(f"  - {file}")

Evaluating:   0%|          | 0/235 [00:00<?, ?it/s]

Hard 测试集结果：
准确率: 0.8756
F1 值:  0.9270
召回率: 0.9634
精确率: 0.8933
模型权重已保存至 /kaggle/working/scam_fraud_model_hard/pytorch_model.bin
分词器已保存至 /kaggle/working/scam_fraud_model_hard

保存的文件列表：
  - pytorch_model.bin
  - tokenizer_config.json
  - tokenizer.json


In [9]:
# ==================== 单元格 8：Hard 阈值分析 ====================
hard_model.eval()
all_probs = []
all_labels = []

with torch.no_grad():
    for batch in hard_test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        features = batch['features'].to(device)
        labels = batch['labels'].to(device)

        logits = hard_model(input_ids, attention_mask, features)
        probs = F.softmax(logits, dim=1)
        all_probs.append(probs.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_probs = np.vstack(all_probs)
fraud_probs = all_probs[:, 1]   # 欺诈概率
binary_true = np.array(all_labels)

from sklearn.metrics import precision_recall_curve
precision, recall, thresholds = precision_recall_curve(binary_true, fraud_probs)

# 最佳阈值（F1最大）
f1_scores = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-9)
best_idx = np.argmax(f1_scores)
best_thr = thresholds[best_idx]
print(f"Hard 最佳阈值（F1最大）: {best_thr:.4f}, F1: {f1_scores[best_idx]:.4f}")

# 固定召回率 ≥ 0.95 的最大阈值（Hard 下召回率可能达不到 0.98）
target_recall = 0.95
valid_idx = np.where(recall[:-1] >= target_recall)[0]
if len(valid_idx) > 0:
    thr_recall = thresholds[valid_idx[-1]]
    print(f"Hard 满足召回率≥{target_recall}的最大阈值: {thr_recall:.4f}")
else:
    thr_recall = 0.5
    print(f"警告：Hard 下无法达到召回率 {target_recall}，使用默认阈值 0.5")

Hard 最佳阈值（F1最大）: 0.9231, F1: 0.9276
Hard 满足召回率≥0.95的最大阈值: 0.9985


In [10]:
# ==================== 单元格 9：加载 Hard 小模型 & 配置大模型 ====================
# 小模型加载
tokenizer = BertTokenizer.from_pretrained(save_path)
hard_model_loaded = BertWithFeatures(MODEL_NAME, FEATURE_DIM, NUM_LABELS)
hard_model_loaded.load_state_dict(torch.load(os.path.join(save_path, 'pytorch_model.bin'), map_location=device))
hard_model_loaded.to(device)
hard_model_loaded.eval()
print("Hard 小模型加载完成！")

# 特征提取函数（复用之前的正则）
def extract_features(text):
    has_url = 1 if url_pattern.search(str(text)) else 0
    has_email = 1 if email_pattern.search(str(text)) else 0
    has_phone = 1 if phone_pattern.search(str(text)) else 0
    return np.array([has_url, has_email, has_phone], dtype=np.float32)

# 大模型配置（阿里云百炼）
!pip install -q openai
from openai import OpenAI

DASHSCOPE_API_KEY = "sk-8d7ac5822cb144aaa8596e2070af8fdf"   # ← 务必替换
client = OpenAI(
    api_key=DASHSCOPE_API_KEY,
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

# 设置默认的安全护栏参数，应用到每一次调用
client_default_extra_body = {
    "data_inspection": {
        "input": "enable",
        "output": "enable"
    }
}
print("安全护栏已启用，后续调用将自动携带此参数。")

def llm_analyze(text):
    prompt = f"""你是一个专业的电信诈骗检测专家。请仔细分析以下对话内容，判断它是否属于诈骗。

**重要提示**：许多正常对话也可能包含“教务处”、“客服”、“会员”、“欠费”等词汇，或者朋友间的玩笑威胁。请务必结合对话的**具体细节、身份可验证性、是否存在索要敏感信息或诱导转账**来综合判断。如果没有明确证据表明是诈骗，请倾向于判断为“正常”。

以下是一个容易被误判为诈骗的正常对话示例：
---
案例（正常催缴班费）：
对话内容："张婷同学你好，我是班长李华。咱们班这学期的班费还没交，麻烦你方便的时候转给我，微信或支付宝都行。"
判断：正常
理由：虽然涉及金钱，但身份明确（班长李华）、关系熟悉（同班同学）、用途合理（班费），且没有威胁、紧迫感或索要验证码等异常行为，属于正常社交沟通。
---

现在请分析以下对话内容：
"{text}"

请用以下格式输出：
【判断】：诈骗 / 正常
【理由】：(简要说明)
"""
    try:
        response = client.chat.completions.create(
            model="qwen3-Flash",
            messages=[
                {"role": "system", "content": "你是一个严谨的电信诈骗检测助手。请综合上下文和细节判断，避免仅凭个别词汇过度反应。若无明确诈骗证据，倾向于判断为正常。"},
                {"role": "user", "content": prompt}
            ],
            temperature=0.1,
            max_tokens=250,
            extra_body=client_default_extra_body
        )
        reply = response.choices[0].message.content.strip()
        if "【判断】：诈骗" in reply:
            return "fraud", reply
        else:
            return "normal", reply
    except Exception as e:
        error_msg = str(e)
        if 'data_inspection_failed' in error_msg:
            print(f"安全审核拦截，暂时判为欺诈: {error_msg[:100]}...")
            return "fraud", error_msg
        else:
            print(f"LLM调用失败: {error_msg}")
            return "error", error_msg

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Hard 小模型加载完成！
安全护栏已启用，后续调用将自动携带此参数。


In [11]:
# ==================== 单元格 10：Hard 协作预测函数 ====================
def hard_small_model_predict(text):
    features = extract_features(text)
    encoding = tokenizer(
        text,
        add_special_tokens=True,
        max_length=MAX_LEN,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    features_tensor = torch.tensor(features).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = hard_model_loaded(input_ids, attention_mask, features_tensor)
        probs = F.softmax(logits, dim=1).cpu().numpy()[0]
    return probs[1], probs   # 欺诈概率，完整概率

def hard_collaborative_predict(text, threshold=0.5):
    fraud_prob, _ = hard_small_model_predict(text)
    if fraud_prob < threshold:
        return "normal", fraud_prob, "小模型", None
    else:
        llm_result, llm_reason = llm_analyze(text)
        return llm_result, fraud_prob, "大模型", llm_reason

In [12]:
# ==================== 断点续传 + 分批测试 ====================
import time
import os

BATCH_SIZE = 1000      # 每批处理 1000 条
THRESHOLD = 0.7
total_samples = len(hard_test_texts)

# 断点续传：查找已有的结果文件
output_path = '/kaggle/working/hard_collaboration_full.csv'
if os.path.exists(output_path):
    existing_df = pd.read_csv(output_path)
    start_idx = len(existing_df)
    all_results = existing_df.to_dict('records')
    print(f"检测到已有 {start_idx} 条结果，续传从索引 {start_idx} 开始...")
else:
    start_idx = 0
    all_results = []

for batch_start in range(start_idx, total_samples, BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE, total_samples)
    batch_texts = hard_test_texts[batch_start:batch_end]
    batch_labels = hard_test_labels[batch_start:batch_end]

    for i, text in enumerate(tqdm(batch_texts, desc=f"Batch {batch_start//BATCH_SIZE+1}")):
        try:
            final_pred, fraud_prob, decision_by, llm_reason = hard_collaborative_predict(text, THRESHOLD)
        except Exception as e:
            final_pred, decision_by, llm_reason = "error", "大模型", str(e)

        sp, _ = hard_small_model_predict(text)
        small_pred = 1 if sp >= 0.5 else 0

        all_results.append({
            'text': text[:50] + "...",
            'true_label': batch_labels[i],
            'final_pred': 1 if final_pred == 'fraud' else 0,
            'small_pred': small_pred,
            'fraud_prob': fraud_prob if 'fraud_prob' in dir() else -1,
            'decision_by': decision_by,
            'llm_reason': llm_reason
        })

    # 每跑完一个批次就保存一次
    pd.DataFrame(all_results).to_csv(output_path, index=False)
    print(f"批次完成，已保存 {len(all_results)} 条结果至 {output_path}")

print(f"全量测试完成！最终结果: {len(all_results)} 条")

Batch 1:   0%|          | 0/1000 [00:00<?, ?it/s]

LLM调用失败: Error code: 404 - {'error': {'message': 'The model `qwen3-Flash` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}, 'request_id': '733aafda-0dc8-9241-9e2e-7aa5821ccb4b'}
LLM调用失败: Error code: 404 - {'error': {'message': 'The model `qwen3-Flash` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}, 'request_id': '4ee16b92-76ad-9549-bf99-037f7819dcaa'}
LLM调用失败: Error code: 404 - {'error': {'message': 'The model `qwen3-Flash` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}, 'request_id': '53a11ed7-98d1-9e8d-ade6-7e5c38b18bb6'}
LLM调用失败: Error code: 404 - {'error': {'message': 'The model `qwen3-Flash` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}, 'request_id': '96857dd7-c88c-97e1-9718

Batch 2:   0%|          | 0/1000 [00:00<?, ?it/s]

LLM调用失败: Error code: 404 - {'error': {'message': 'The model `qwen3-Flash` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}, 'request_id': 'c0a4c030-158b-93d0-9e17-d4a3e3f7faa0'}
LLM调用失败: Error code: 404 - {'error': {'message': 'The model `qwen3-Flash` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}, 'request_id': 'c9b2f9e1-c107-973d-ab77-80cc154dcd2d'}
LLM调用失败: Error code: 404 - {'error': {'message': 'The model `qwen3-Flash` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}, 'request_id': 'a5f2f636-37da-9167-bfa5-d73984d2ab59'}
LLM调用失败: Error code: 404 - {'error': {'message': 'The model `qwen3-Flash` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}, 'request_id': 'c8a24a2f-6ea9-9dae-8013

Batch 3:   0%|          | 0/1000 [00:00<?, ?it/s]

...

LLM调用失败: Error code: 404 - {'error': {'message': 'The model `qwen3-Flash` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}, 'request_id': 'f247c827-d848-9f86-ad11-5b1d066d3c89'}
LLM调用失败: Error code: 404 - {'error': {'message': 'The model `qwen3-Flash` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}, 'request_id': '15313e0c-8f31-9e8f-ae5a-0474f8c47f82'}
LLM调用失败: Error code: 404 - {'error': {'message': 'The model `qwen3-Flash` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}, 'request_id': '9916ff7c-b018-946a-be5f-0b362d0ccffd'}
LLM调用失败: Error code: 404 - {'error': {'message': 'The model `qwen3-Flash` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}, 'request_id': 'b4529b8e-a06f-9cd9-bd5b

Batch 7:   0%|          | 0/1000 [00:00<?, ?it/s]

LLM调用失败: Error code: 404 - {'error': {'message': 'The model `qwen3-Flash` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}, 'request_id': '6660d1fb-6c92-9165-a877-7746bcca5b6c'}
LLM调用失败: Error code: 404 - {'error': {'message': 'The model `qwen3-Flash` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}, 'request_id': '18f7012e-acc8-92de-9847-fa6f96d0e478'}
LLM调用失败: Error code: 404 - {'error': {'message': 'The model `qwen3-Flash` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}, 'request_id': '696b5a59-4878-982c-ac50-604821fca933'}
LLM调用失败: Error code: 404 - {'error': {'message': 'The model `qwen3-Flash` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}, 'request_id': '6fcbae73-eb39-978f-936f

Batch 8:   0%|          | 0/500 [00:00<?, ?it/s]

LLM调用失败: Error code: 404 - {'error': {'message': 'The model `qwen3-Flash` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}, 'request_id': '35fdd731-c764-9852-9655-4f7c8ff1af49'}
LLM调用失败: Error code: 404 - {'error': {'message': 'The model `qwen3-Flash` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}, 'request_id': '481da21a-a6fd-9953-838b-862a94d597f9'}
LLM调用失败: Error code: 404 - {'error': {'message': 'The model `qwen3-Flash` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}, 'request_id': '9d60462b-1b3c-9ea3-86cc-2eb1a47047ea'}
LLM调用失败: Error code: 404 - {'error': {'message': 'The model `qwen3-Flash` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}, 'request_id': 'c902cfca-8212-9683-a9fa

In [13]:
# ==================== 单元格 11：Hard 协作实验（Kaggle 版）====================
THRESHOLD = 0.7   # 使用单元格 8 得到的满足召回率阈值
N_TEST = 50           # 先测 200 条，全量可改为 len(hard_test_texts)
sample_texts = hard_test_texts[:N_TEST]
sample_labels = hard_test_labels[:N_TEST]

results = []
small_preds = []
llm_calls = 0

print(f"开始 Hard 协作检测，阈值 = {THRESHOLD:.4f}...")
for text in tqdm(sample_texts):
    start = time.time()
    final_pred, fraud_prob, decision_by, llm_reason = hard_collaborative_predict(text, THRESHOLD)
    elapsed = time.time() - start

    sp, _ = hard_small_model_predict(text)
    small_pred = 1 if sp >= 0.5 else 0
    small_preds.append(small_pred)

    if decision_by == "大模型":
        llm_calls += 1

    results.append({
        'text': text[:50] + "...",
        'true_label': sample_labels[len(results)],
        'final_pred': 1 if final_pred == 'fraud' else 0,
        'small_pred': small_pred,
        'fraud_prob': fraud_prob,
        'decision_by': decision_by,
        'llm_reason': llm_reason
    })

final_preds = [r['final_pred'] for r in results]

print("\n========== Hard 实验结果 ==========")
print(f"测试样本数: {len(sample_texts)}")
print(f"大模型调用次数: {llm_calls} ({llm_calls/len(sample_texts)*100:.1f}%)")
print("\n--- 小模型独立 ---")
print(f"准确率: {accuracy_score(sample_labels, small_preds):.4f}")
print(f"F1 值:  {f1_score(sample_labels, small_preds):.4f}")
print(f"召回率: {recall_score(sample_labels, small_preds):.4f}")
print("\n--- 大小模型协作 ---")
print(f"准确率: {accuracy_score(sample_labels, final_preds):.4f}")
print(f"F1 值:  {f1_score(sample_labels, final_preds):.4f}")
print(f"召回率: {recall_score(sample_labels, final_preds):.4f}")

# 保存结果到 Kaggle 工作目录
hard_results_df = pd.DataFrame(results)
save_csv_path = '/kaggle/working/hard_collaboration_results.csv'
hard_results_df.to_csv(save_csv_path, index=False)
print(f"\nHard 详细结果已保存至 {save_csv_path}")

开始 Hard 协作检测，阈值 = 0.7000...


  0%|          | 0/50 [00:00<?, ?it/s]

LLM调用失败: Error code: 404 - {'error': {'message': 'The model `qwen3-Flash` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}, 'request_id': '7f8d98db-b627-96ff-a964-e17091c0fec9'}
LLM调用失败: Error code: 404 - {'error': {'message': 'The model `qwen3-Flash` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}, 'request_id': '46196905-6811-971e-bde7-4a1915f2803c'}
LLM调用失败: Error code: 404 - {'error': {'message': 'The model `qwen3-Flash` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}, 'request_id': '284ca4de-7a1a-939a-b6a8-c690072f5b03'}
LLM调用失败: Error code: 404 - {'error': {'message': 'The model `qwen3-Flash` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}, 'request_id': '05daff61-e7c0-9f32-b5d9

In [14]:
# ==================== 错误样本分析 ====================
# 将 results 转为 DataFrame（如果还没有的话）
results_df = pd.DataFrame(results)

# 筛选大模型参与判断的样本（decision_by == '大模型'）
llm_involved = results_df[results_df['decision_by'] == '大模型']

# 找出大模型判断错误的样本
llm_errors = llm_involved[llm_involved['true_label'] != llm_involved['final_pred']]

# 分类错误类型
false_negatives = llm_errors[llm_errors['true_label'] == 1]  # 真实欺诈，判为正常
false_positives = llm_errors[llm_errors['true_label'] == 0]  # 真实正常，判为欺诈

print(f"大模型参与判断的样本数: {len(llm_involved)}")
print(f"大模型错误样本数: {len(llm_errors)}")
print(f"  - 假阴性 (欺诈被漏判): {len(false_negatives)}")
print(f"  - 假阳性 (正常被误判): {len(false_positives)}")

# 显示前 3 条假阴性样本（欺诈漏报）
print("\n=== 假阴性示例（欺诈被大模型漏判）===")
for idx, row in false_negatives.head(3).iterrows():
    print(f"文本: {row['text']}...")
    print(f"小模型预测: {row['small_pred']} | 大模型预测: {row['final_pred']}")
    print(f"大模型理由: {row['llm_reason'][:150]}...\n")

# 显示前 3 条假阳性样本（正常被误判为欺诈）
print("\n=== 假阳性示例（正常被大模型误判为欺诈）===")
for idx, row in false_positives.head(10).iterrows():
    print(f"文本: {row['text']}...")
    print(f"小模型预测: {row['small_pred']} | 大模型预测: {row['final_pred']}")
    print(f"大模型理由: {row['llm_reason'][:150]}...\n")

大模型参与判断的样本数: 42
大模型错误样本数: 38
  - 假阴性 (欺诈被漏判): 38
  - 假阳性 (正常被误判): 0

=== 假阴性示例（欺诈被大模型漏判）===
文本: 您好你好，我是梅州市防疫部门的工作人员 你的行程流调码上显示，你跟一例阳性病例的行程有交集 我给你散......
小模型预测: 1 | 大模型预测: 0
大模型理由: Error code: 404 - {'error': {'message': 'The model `qwen3-Flash` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'p...

文本: 很抱歉打扰到您，我是申通快递的客服，联络您呢是因为 不好意思通知您，您的快递不见了，真是非常抱歉，我......
小模型预测: 1 | 大模型预测: 0
大模型理由: Error code: 404 - {'error': {'message': 'The model `qwen3-Flash` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'p...

文本: 小张啊！ 哎哟，你咋连我声音都听不出来了？您好好想想，看看这段时间都联络谁了，我们约好了嘛。 昨天晚......
小模型预测: 1 | 大模型预测: 0
大模型理由: Error code: 404 - {'error': {'message': 'The model `qwen3-Flash` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'p...


=== 假阳性示例（正常被大模型误判为欺诈）===
